# 653. Two Sum IV - Input is a BST

[Problem](https://leetcode.com/problems/two-sum-iv-input-is-a-bst/) · difficulty: easy

Six approaches spanning 27 ms to 0 ms on the judge. Five of them are the same algorithm; this
notebook separates the two things that actually differ — **what holds the complements** and
**what order the walk visits nodes** — and measures each on its own.


In [ ]:
import pathlib, sys

ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / 'lc').is_dir())
PROBLEM = ROOT / 'problems' / '0653-two-sum-iv-input-is-a-bst'
sys.path.insert(0, str(ROOT))

from lc.harness import load_solutions, load_module

solutions = load_solutions(PROBLEM)
TreeNode = load_module(PROBLEM / 'solutions.py').TreeNode
[s.__name__ for s in solutions]


## Summary

| Approach | Time | Space | Holds complements | Order |
|---|---|---|---|---|
| `SolutionRecursiveList` | O(n²) | O(n) | list | depth-first |
| `SolutionStackSet` | O(n) | O(n) | set | depth-first |
| `SolutionRecursiveSetSeeded` | O(n) | O(n) | set | depth-first |
| `SolutionRecursiveSet` | O(n) | O(n) | set | depth-first |
| `SolutionSearchPerNode` | O(n·h) | O(h) | nothing — searches the BST | depth-first |
| `SolutionQueueSet` | O(n) | O(n) | set | breadth-first |


## The trick, step by step

Visiting a node, record the value that *would* complete a pair with it. A later node whose own
value is already recorded closes the pair. Recording after the check is what stops a node from
pairing with itself — the same rule as in problem 1, now over a traversal instead of an array.


In [ ]:
def trace(values, k):
    seen = set()
    for value in values:
        hit = value in seen
        print(f'visit {value:>3}  seen={sorted(seen)}  waiting for {value}? {hit}')
        if hit:
            return True
        seen.add(k - value)
    return False

print(trace([5, 3, 2, 4, 6, 7], 9))


## Building the trees

`bst` makes a height-balanced tree from sorted values, which is the shape the measurements use.


In [ ]:
def bst(values):
    if not values:
        return None
    middle = len(values) // 2
    return TreeNode(values[middle], bst(values[:middle]), bst(values[middle + 1:]))

TREE = bst(list(range(1, 8192)))
NO_PAIR = 100000            # nothing sums to this, so every node must be visited
PAIR_EXISTS = 1 + 8191      # many pairs reach it, the nearest straddling the root
print('nodes:', 8191)


## Where the time goes

Two scenarios, because they answer different questions. With no pair every approach must visit
every node, so the measurement is pure per-node cost. With a pair available, the traversal order
decides how soon the walk stumbles on it.


In [ ]:
import time

def timed(solution, tree, k, runs=9):
    """Minimum of `runs` timings: single runs vary by more than the effects here."""
    samples = []
    for _ in range(runs):
        start = time.perf_counter()
        solution().findTarget(tree, k)
        samples.append((time.perf_counter() - start) * 1e3)
    return min(samples)

print(f"{'approach':<30}{'no pair (full walk)':>22}{'pair exists':>16}")
for solution in solutions:
    print(f'{solution.__name__:<30}'
          f'{timed(solution, TREE, NO_PAIR):19.2f} ms'
          f'{timed(solution, TREE, PAIR_EXISTS):13.2f} ms')


`SolutionRecursiveList` is two orders of magnitude off the rest, and it differs from
`SolutionRecursiveSetSeeded` by one word: `seen` is a list instead of a set. `in` on a list is a
scan, so the walk is O(n²). That is the whole gap — not the recursion, not the seeding.


## Is the queue actually the reason?

`SolutionQueueSet` beats `SolutionStackSet` on the judge, 0 ms against 8 ms, and it is tempting to
credit breadth-first order. But the two differ in three ways at once: order, whether `None`
children are pushed, and `for node in queue` against `while stack: stack.pop()`.

Two controls separate them — one keeps depth-first order but stops pushing `None`, the other is
breadth-first with an ordinary `deque.popleft`.


In [ ]:
from collections import deque

class StackNoNone:
    """Depth-first, like SolutionStackSet, but children are filtered before pushing."""

    def findTarget(self, root, k):
        stack, seen = [root], set()
        while stack:
            node = stack.pop()
            if node.val in seen:
                return True
            seen.add(k - node.val)
            if node.left:
                stack.append(node.left)
            if node.right:
                stack.append(node.right)
        return False


class QueueDeque:
    """Breadth-first, like SolutionQueueSet, but with a real deque and popleft."""

    def findTarget(self, root, k):
        queue, seen = deque([root]), set()
        while queue:
            node = queue.popleft()
            if node.val in seen:
                return True
            seen.add(k - node.val)
            if node.left:
                queue.append(node.left)
            if node.right:
                queue.append(node.right)
        return False


by_name = {s.__name__: s for s in solutions}
controls = {
    'SolutionStackSet (depth, pushes None)': by_name['SolutionStackSet'],
    'StackNoNone     (depth, no None)': StackNoNone,
    'QueueDeque      (breadth, popleft)': QueueDeque,
    'SolutionQueueSet (breadth, growing list)': by_name['SolutionQueueSet'],
}
for label, cls in controls.items():
    print(f'{label:<44}{timed(cls, TREE, NO_PAIR, runs=15):8.2f} ms')


Reading the ladder: not pushing children that do not exist buys about 0.19 ms, iterating the
growing list instead of popping buys about 0.16 ms, and switching depth-first to breadth-first
buys **roughly nothing**. The queue is not why `SolutionQueueSet` is quick on a full walk.

This only became visible with repeated timings — the first attempt at this comparison used a
single run per variant and put the controls in the opposite order.

Order does earn its keep when a pair *exists*: depth-first commits to one side of the tree
before it can see the other, while breadth-first holds both subtrees after two levels — and in a
BST a pair summing to a mid-range `k` usually straddles the root. The second column of the
measurement above is that effect, not this one.


## Takeaway

- The container is a complexity decision (list → O(n²), set → O(n)); the traversal order is a
  constant-factor one. Only the first is worth arguing about before measuring.
- When two implementations differ in three ways, a benchmark comparing them measures nothing in
  particular. Controls that change one thing at a time are the cheap fix.
- `SolutionSearchPerNode` is the only approach using the BST ordering, and it is both the
  lightest on memory and the fastest when a pair exists early — and the slowest of the linear
  family when none does. The right choice depends on the input distribution, which on LeetCode
  is invisible.
